The goal of this project is to experiment with YOLO and learn what we can do with it. We can also try to reinforce the application with the use of TensorRT given that this model will be running on a Jetson AGX Orin, or theoretically any parallel compute

# Imports

In [1]:
import time
from pathlib import Path
import cv2
import numpy as np
import torch
from ultralytics import YOLO

print(f"torch {torch.__version__}, cuda available: {torch.cuda.is_available()}")

torch 2.2.2, cuda available: False


# Dataset - FSOCO Sample

In [2]:
%pip install -q remotezip


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import random
from pathlib import Path

from remotezip import RemoteZip

FSOCO_BBOX_URL = "http://fsoco.cs.uni-freiburg.de/datasets/fsoco_bounding_boxes_train.zip"
SAMPLE_SIZE = 200        # number of images to pull — tune as needed
RANDOM_SEED = 42
OUT_DIR = Path("../../ml/data/fsoco_sample")

with RemoteZip(FSOCO_BBOX_URL) as zf:
    names = zf.namelist()

print(f"{len(names)} entries in the remote archive")
print(*names[:10], sep="\n")

23145 entries in the remote archive
meta.json
mms/ann/mms_00614.jpg.json
mms/ann/mms_00159.jpg.json
mms/ann/mms_00624.jpg.json
mms/ann/mms_00037.jpg.json
mms/ann/mms_00554.jpg.json
mms/ann/mms_00262.jpg.json
mms/ann/mms_00520.jpg.json
mms/ann/mms_00246.jpg.json
mms/ann/mms_00351.jpg.json


Check the printed paths above against what the code assumes.

In [4]:
meta_candidates = [n for n in names if n.endswith("meta.json")]
print(meta_candidates)

with RemoteZip(FSOCO_BBOX_URL) as zf:
    meta = json.loads(zf.read(meta_candidates[0]))

# Pinned explicitly (rather than derived from meta.json's order) so class
# indices are stable across reruns. Drops unknown_cone (FSOCO's own docs
# call it non-rules-compliant) and the seg_* classes (segmentation-only,
# never appear on rectangle objects in this bounding-box archive).
classes = ["blue_cone", "yellow_cone", "orange_cone", "large_orange_cone"]
print("Using classes:", classes)
print("(available in meta.json:", [c["title"] for c in meta["classes"]], ")")

['meta.json']
Using classes: ['blue_cone', 'yellow_cone', 'orange_cone', 'large_orange_cone']
(available in meta.json: ['seg_orange_cone', 'unknown_cone', 'yellow_cone', 'seg_large_orange_cone', 'seg_blue_cone', 'seg_unknown_cone', 'seg_yellow_cone', 'blue_cone', 'orange_cone', 'large_orange_cone'] )


Let's take a look at the images and how they're organized in FSOCO

In [5]:
img_entries = [n for n in names if "/img/" in n and not n.endswith("/")]
print(f"{len(img_entries)} total images in archive")

random.seed(RANDOM_SEED)
sample_imgs = random.sample(img_entries, min(SAMPLE_SIZE, len(img_entries)))

def ann_path_for(img_path: str) -> str:
    return img_path.replace("/img/", "/ann/") + ".json"

sample_pairs = [(p, ann_path_for(p)) for p in sample_imgs]
sample_pairs[:5]

11572 total images in archive


[('ecurieaix/img/ecurieaix_00340.png',
  'ecurieaix/ann/ecurieaix_00340.png.json'),
 ('uop/img/BME_01335.jpg', 'uop/ann/BME_01335.jpg.json'),
 ('mms/img/mms_00358.jpg', 'mms/ann/mms_00358.jpg.json'),
 ('prom/img/prom_00304.jpg', 'prom/ann/prom_00304.jpg.json'),
 ('pwrrt/img/pwrrt_00196.png', 'pwrrt/ann/pwrrt_00196.png.json')]

In [6]:
missing = [ann for _, ann in sample_pairs if ann not in names]
print(f"{len(missing)} / {len(sample_pairs)} annotation paths not found")
assert not missing, "ann_path_for() doesn't match this archive's layout — check the printed names above."

0 / 200 annotation paths not found


In [7]:
(OUT_DIR / "images").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "labels").mkdir(parents=True, exist_ok=True)

class_to_id = {name: i for i, name in enumerate(classes)}
(OUT_DIR / "classes.txt").write_text("\n".join(classes))

def supervisely_to_yolo(ann: dict) -> list[str]:
    h, w = ann["size"]["height"], ann["size"]["width"]
    lines = []
    for obj in ann["objects"]:
        if obj["geometryType"] != "rectangle":
            continue
        if obj["classTitle"] not in class_to_id:
            continue  # drops unknown_cone and any stray seg_* class
        (x1, y1), (x2, y2) = obj["points"]["exterior"]
        cx, cy = (x1 + x2) / 2 / w, (y1 + y2) / 2 / h
        bw, bh = abs(x2 - x1) / w, abs(y2 - y1) / h
        cls_id = class_to_id[obj["classTitle"]]
        lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    return lines

with RemoteZip(FSOCO_BBOX_URL) as zf:
    for img_path, ann_path in sample_pairs:
        img_bytes = zf.read(img_path)
        ann = json.loads(zf.read(ann_path))

        out_name = Path(img_path).name
        (OUT_DIR / "images" / out_name).write_bytes(img_bytes)

        label_name = Path(out_name).with_suffix(".txt").name
        (OUT_DIR / "labels" / label_name).write_text("\n".join(supervisely_to_yolo(ann)))

print(f"Wrote {len(sample_pairs)} image/label pairs to {OUT_DIR}")

Wrote 200 image/label pairs to ../../ml/data/fsoco_sample


In [8]:
dataset_yaml = OUT_DIR / "dataset.yaml"
dataset_yaml.write_text(
    f"path: {OUT_DIR.resolve()}\n"
    f"train: images\n"
    f"val: images  # same dir for now — do a real split before actual training\n"
    f"names:\n" + "\n".join(f"  {i}: {name}" for i, name in enumerate(classes)) + "\n"
)
print(dataset_yaml.read_text())

path: /Users/songyueli/Documents/GitHub/shabang/formula_driverless/ml/data/fsoco_sample
train: images
val: images  # same dir for now — do a real split before actual training
names:
  0: blue_cone
  1: yellow_cone
  2: orange_cone
  3: large_orange_cone



# Baseline COCO Weights

In [9]:
model = YOLO("yolo26n.pt") # downloads pretrained weights
test_img = next((OUT_DIR / "images").glob("*.jpg"))
print(test_img)
results = model(str(test_img))
results[0].show()

../../ml/data/fsoco_sample/images/prom_00076.jpg

image 1/1 /Users/songyueli/Documents/GitHub/shabang/formula_driverless/perception/notebooks/../../ml/data/fsoco_sample/images/prom_00076.jpg: 416x640 (no detections), 90.8ms
Speed: 1.9ms preprocess, 90.8ms inference, 0.4ms postprocess per image at shape (1, 3, 416, 640)


# Train fine-tuned cone model

Runs the actual training via `ml/train.py`, against the FSOCO sample from
above. `epochs`/`imgsz` here are a fast CPU smoke test, not a real training
config — bump them up (and get a bigger sample / real GPU) once you're doing
an actual training run. Also note `train == val` in `dataset.yaml` right now
(no real split), so any validation metrics from this run are meaningless —
this is purely to confirm the pipeline produces a loadable `best.pt`.

In [10]:
import sys
sys.path.append(str(Path("../../ml").resolve()))
from train import train

train(
    data_yaml=str(OUT_DIR / "dataset.yaml"),
    model_variant="yolo26n.pt",
    profile="auto",  # picks "smoke" (CPU) or "full" (CUDA, e.g. the Jetson) automatically
    project=str(Path("../../ml/runs/detect").resolve()),
    name="train",
)

[train] profile=smoke (cuda available: False) -> epochs=5, imgsz=640
New https://pypi.org/project/ultralytics/8.4.117 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.115 🚀 Python-3.11.15 torch-2.2.2 CPU (Intel Core i9-9880H 2.30GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../ml/data/fsoco_sample/dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_widt

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1))
          (act): SiLU(inplace=True)
        )
        (m): ModuleList(
          (0): Bottleneck(
            (cv1): Conv(
              (conv): Conv2d(16, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
              (act): SiLU(inplace=True)
            )
            (cv2): Conv(
              (conv): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
              (act): SiLU(inplace=True)
    

# Load fine-tuned cone model

In [11]:
cone_model = YOLO("../../ml/runs/detect/train/weights/best.pt")
cone_model.names # expect {0: blue, 1: yellow, 2: orange, 3: large_orange}

results = cone_model(str(test_img))
results[0].show()


image 1/1 /Users/songyueli/Documents/GitHub/shabang/formula_driverless/perception/notebooks/../../ml/data/fsoco_sample/images/prom_00076.jpg: 416x640 (no detections), 159.2ms
Speed: 3.1ms preprocess, 159.2ms inference, 0.3ms postprocess per image at shape (1, 3, 416, 640)


# Inference latency benchmarking

In [12]:
def benchmark(model, img, n=50, warmup=5, **predict_kwargs):
    for _ in range(warmup):
        model(img, verbose=False, **predict_kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(n):
        model(img, verbose=False, **predict_kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    return elapsed / n * 1000

ms = benchmark(cone_model, str(test_img))
print(f"{ms:.2f} ms/frame  ({1000/ms:.1f} FPS)")

137.52 ms/frame  (7.3 FPS)


# Model size sweep - Accuracy vs latency

In [13]:
variants = ["yolo26n.pt", "yolo26s.pt", "yolo26m.pt"] 
for v in variants:
    m = YOLO(v)
    ms = benchmark(m, str(test_img))
    print(f"{v:15s} {ms:6.2f} ms/frame  ({1000/ms:5.1f} FPS)")

yolo26n.pt      141.85 ms/frame  (  7.0 FPS)
yolo26s.pt      258.21 ms/frame  (  3.9 FPS)
yolo26m.pt      542.54 ms/frame  (  1.8 FPS)


# Precision and Export Optimizations

In [ ]:
# FP16
ms_fp16 = benchmark(cone_model, str(test_img), quantize=16) # GPU-only — expect no real speedup on this CPU-only Mac
print(f"FP16: {ms_fp16:.2f} ms/frame  ({1000/ms_fp16:.1f} FPS)")

# ONNX export
cone_model.export(format="onnx", imgsz=1440, simplify=True)

import onnxruntime as ort
sess = ort.InferenceSession("../../ml/runs/detect/train/weights/best.onnx",
                            providers = ["CPUExecutionProvider"]) # or CUDAExecutionProvider

# build an onnxruntime benchmark loop analogous to 'benchmark()' above

# Results & Analysis